# WMM Laboratorium #7 - 18.05.2026, grupa 101 - 16:15
## Autor: Kacper Skrodzki


## Wstęp

Celem tych laboratoriów jest zaznajomienie się ze sposobami detekcji twarzy przy użyciu różnych detektorów, problemami które utrudniają detekcję i parametrami które wpływaja na nią. W czasie laboratoriów zostały użyte cztery detektory:
- Kaskada Haara *(z biblioteki OpenCV)*
- Histogram zorientowanych gradientów *(HOG)* z maszyną wektorów nośnych *(SVM)* *(z biblioteki Dlib)*
- Splotowa sieć neuronowa *(MMOD z biblioteki Dlib)*
- InsightFace

Obrazy testowe i ich **GT *(ground truth)*** pochodzą z bazy zdjęć WIDER.

## Funkcje pomocnicze i miary efektywności

W celu ocenienia sprawczości działania danych detektorów przyjąłem następujące miary efektywności:
- Wykryte obiekty - ilość wszystkich detekcji
- Prawdziwe twarze w GT - ilość wszytskich twarzy na obrazie
- Trafienia *(TP - True Positivies)* - ilość dobrych detekcji
- Fałszywe *(FP - False Positives)* - ilóść niepoprawnych detekcji (brak twarzy, za duży obszar, parę twarzy w jednym)
- Przeoczone *(FN - False Negativies)* - ilość ominiętych twarzych
- Precyzja (w %) - stosunek Trafień do Fałszywych detekcji *(TP / FP)*
- Czułość/Recall (%) - stosunek Trafień do liczby wszystkich twarzy w GT

Fałszywe detekcje *(FP)* są ustalane przy pomocy informacji z GT - do znalezionego przez detektor obszaru szukamy pasującego obaszaru w GT. Jeśli oba obszary pokrywają się nawazajem w 50 % to uznajemy tą detekcję za Trafienie. W przeciwnym razie detekcja jest klasyfikowana jako Fałszywe.

Do usprawnienia ćwiczenia utworzono dwie funkcje pomocnicze: 
- *load_gt()* - wczytanie GT wszystkich zdjęć z archiwum i zapisanie ich w słowniku
- *evaluate_with_gt* - klasyfikacja wyników detektora

In [ ]:
import cv2
import dlib
import insightface

from PIL import Image
from IPython.display import display
import pandas as pd
from enum import Enum
import numpy as np

In [ ]:
def load_gt(txt_filepath):
    """
    Funkcja wczytuje plik tekstowy z Ground Truth w formacie WIDER FACE
    i zwraca słownik: { 'nazwa_pliku.jpg': [[x1, y1, x2, y2], ...], ... }
    """
    gt_dict = {}

    with open(txt_filepath, 'r') as f:
        lines = f.readlines()

    i = 0
    while i < len(lines):
        line = lines[i].strip()
        # Pomijanie pustych linii
        if not line:
            i += 1
            continue

        # 1. Wczytanie nazwy pliku
        filename = line
        i += 1

        # 2. Wczytanie liczby twarzy
        num_boxes = int(lines[i].strip())

        boxes = []
        # W zbiorze WIDER FACE jest taki specyficzny wyjątek:
        # jeśli num_boxes == 0, to w kolejnej linijce i tak są zera (0 0 0 0 0 0 0 0 0 0)
        if num_boxes == 0:
            i += 1
        else:
            # 3. Wczytanie współrzędnych dla każdej twarzy
            for _ in range(num_boxes):
                i += 1
                parts = lines[i].strip().split()

                # Pierwsze 4 wartości to x, y, width, height
                x = int(parts[0])
                y = int(parts[1])
                w = int(parts[2])
                h = int(parts[3])

                # Konwersja z [x, y, w, h] na [x1, y1, x2, y2] do obliczeń IoU
                boxes.append([x, y, x + w, y + h])

        gt_dict[filename] = boxes
        i += 1

    return gt_dict

In [ ]:
class DetectorTypes(Enum):
    HAAR = 1
    HOG_SVM = 2
    MMOD = 3
    INSIGHTFACE = 4


def calculate_iou(boxA, boxB):
    """
    Oblicza współczynnik Intersection over Union (IoU) dla dwóch ramek.
    Format ramek: [x1, y1, x2, y2] (lewy górny i prawy dolny róg)
    """
    # Wyznaczenie współrzędnych części wspólnej (przecięcia)
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    # Pole części wspólnej
    interArea = max(0, xB - xA) * max(0, yB - yA)

    # Pola obu ramek
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])

    # IoU = część wspólna / (suma pól - część wspólna)
    if float(boxAArea + boxBArea - interArea) == 0:
        return 0.0
    iou = interArea / float(boxAArea + boxBArea - interArea)

    return iou


def evaluate_with_gt(detections, gt_boxes, detector_type: DetectorTypes, iou_threshold=0.5):
    """
    Funkcja oceniająca detekcje na podstawie Ground Truth (GT).
    gt_boxes: lista ramek referencyjnych w formacie [[x1,y1,x2,y2], [x1,y1,x2,y2], ...]
    """
    # 1. Konwersja detekcji do wspólnego formatu [x1, y1, x2, y2]
    pred_boxes = []
    for det in detections:
        if detector_type == DetectorTypes.HAAR:
            x, y, w, h = det
            pred_boxes.append([x, y, x + w, y + h])
        elif detector_type == DetectorTypes.HOG_SVM:
            pred_boxes.append([det.left(), det.top(), det.right(), det.bottom()])
        elif detector_type == DetectorTypes.MMOD:
            pred_boxes.append([det.rect.left(), det.rect.top(), det.rect.right(), det.rect.bottom()])
        elif detector_type == DetectorTypes.INSIGHTFACE:
            x1, y1, x2, y2 = det.bbox
            pred_boxes.append([int(x1), int(y1), int(x2), int(y2)])

    # Kopie list, żebyśmy mogli "odznaczać" przypisane już twarze GT
    unmatched_gts = list(gt_boxes).copy()

    true_positives = 0  # Trafienia (hit)

    # Dla każdej predykcji szukamy najlepiej pasującej twarzy z GT
    for pred in pred_boxes:
        best_iou = 0
        best_gt_idx = -1

        for idx, gt in enumerate(unmatched_gts):
            iou = calculate_iou(pred, gt)
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = idx

        if best_iou >= iou_threshold:
            true_positives += 1
            # Usuwamy przypisany GT, żeby inna ramka go "nie ukradła" (np. gdy wykryto 2 twarze w 1 miejscu)
            unmatched_gts.pop(best_gt_idx)

    # Obliczamy resztę statystyk
    # False Positives: fałszywe alarmy (wykryte ramki, które nie mają pokrycia z GT)
    false_positives = len(pred_boxes) - true_positives

    # False Negatives: twarze z GT, których detektor w ogóle nie znalazł
    false_negatives = len(unmatched_gts)

    # Obliczenie precyzji (ile wykrytych to faktycznie twarze) i czułości (ile twarzy z GT udało się znaleźć)
    precision = true_positives / len(pred_boxes) if len(pred_boxes) > 0 else 0
    recall = true_positives / len(gt_boxes) if len(gt_boxes) > 0 else 0

    return {
        "Wykryte obiekty": len(pred_boxes),
        "Prawdziwe twarze w GT": len(gt_boxes),
        "Trafienia (TP)": true_positives,
        "Fałszywe (FP)": false_positives,
        "Przeoczone (FN)": false_negatives,
        "Precyzja (%)": round(precision * 100, 2),
        "Czułość/Recall (%)": round(recall * 100, 2)
    }


## Zadanie 1 - porównanie detektorów

In [ ]:
results_ex1 = []
results_all = []

gt_all_images = load_gt("ground_truth_bbx.txt")

image_ex1 = "37--Soccer/37_Soccer_soccer_ball_37_907.jpg"

gt_ex1 = gt_all_images.get(image_ex1, [])

### Obraz testowy

In [ ]:
img = cv2.imread("37_Soccer_soccer_ball_37_907.jpg")
# przygotowanie obrazów: monochromatycznego i RGB (cv2.imread() zwraca obraz w formacie BGR - inna kolejność składowych)
img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

display(Image.fromarray(img_rgb))  # display() wymaga obrazu RGB (w przeciwieństwie do cv2.imshow(), która wymaga BGR)

### Kaskada Haara

In [ ]:
# utworzenie i inicjalizacja detektora
haar_detector = cv2.CascadeClassifier("haarcascade_frontalface_default.xml")

In [ ]:
# wywołanie detektora dla określonego obrazu (img_gray)
# wynikiem jest lista prostokątów w formacie [x, y, width, height]
haar_results = haar_detector.detectMultiScale(img_gray, scaleFactor=1.05, minNeighbors=5, minSize=(30, 30), flags=cv2.CASCADE_SCALE_IMAGE)
res_haar_1 = evaluate_with_gt(haar_results, gt_ex1, detector_type=DetectorTypes.HAAR)
res_haar_1["Detektor"] = "OpenCV (Haar)"
res_haar_1["Parametry"] = "scaleFactor=1.05, minNeighbors=5, minSize=(30, 30)"
results_ex1.append(res_haar_1)
results_all.append(res_haar_1)

# narysowanie wyników na kopii obrazu i wyświetlenie
img_haar = img_rgb.copy()
for (x, y, w, h) in haar_results:
    cv2.rectangle(img_haar, (x, y), (x + w, y + h), (0, 255, 0), 2)
display(Image.fromarray(img_haar))

### Histogram zorientowanych gradientów (HOG) z maszyną wektorów nośnych (SVM)

In [ ]:
# utworzenie i inicjalizacja detektora
hog_svm_detector = dlib.get_frontal_face_detector()

In [ ]:
hog_svm_results = hog_svm_detector(img_rgb, 1)
res_dlib_hog = evaluate_with_gt(hog_svm_results, gt_ex1, detector_type=DetectorTypes.HOG_SVM)
res_dlib_hog["Detektor"] = "Dlib (HOG)"
res_dlib_hog["Parametry"] = "upsample=1"
results_ex1.append(res_dlib_hog)
results_all.append(res_dlib_hog)

img_hog_svm = img_rgb.copy()
for rect in hog_svm_results:
    cv2.rectangle(img_hog_svm, (rect.left(), rect.top()), (rect.right(), rect.bottom()), (0, 255, 0), 2)
display(Image.fromarray(img_hog_svm))

### Splotowa sieć neuronowa (CNN) z biblioteki dlib

In [ ]:
cnn1_detector = dlib.cnn_face_detection_model_v1('mmod_human_face_detector.dat')

In [ ]:
cnn1_results = cnn1_detector(img_rgb, 1)
res_dlib_mmod = evaluate_with_gt(cnn1_results, gt_ex1, detector_type=DetectorTypes.MMOD)
res_dlib_mmod["Detektor"] = "Dlib (MMOD)"
res_dlib_mmod["Parametry"] = "upsample=1"
results_ex1.append(res_dlib_mmod)
results_all.append(res_dlib_mmod)

# narysowanie wyników na kopii obrazu i wyświetlenie
img_cnn1 = img_rgb.copy()
for res in cnn1_results:
    rect = res.rect
    cv2.rectangle(img_cnn1, (rect.left(), rect.top()), (rect.right(), rect.bottom()), (0, 255, 0), 2)
display(Image.fromarray(img_cnn1))

### InsightFace - inna sieć neuronowa

In [ ]:
# utworzenie i inicjalizacja detektora
model_name = 'buffalo_s'  # model: small (_s), medium (_m) lub large (_l)
insf_detector = insightface.app.FaceAnalysis(name=model_name, root='insightface',
                                             allowed_modules=['detection'], providers=['CPUExecutionProvider'])
insf_detector.prepare(ctx_id=0, det_size=(1024, 1024))

In [ ]:
insf_results = insf_detector.get(img)
res_insf = evaluate_with_gt(insf_results, gt_ex1, detector_type=DetectorTypes.INSIGHTFACE)
res_insf["Detektor"] = "InsightFace"
res_insf["Parametry"] = "det_size=(1024, 1024)"
results_ex1.append(res_insf)
results_all.append(res_insf)

img_insf = img_rgb.copy()
for res in insf_results:
    rect = res.bbox.round().astype(int)
    cv2.rectangle(img_insf, (rect[0], rect[1]), (rect[2], rect[3]), (0, 255, 0), 2)
display(Image.fromarray(img_insf))

### Tabela z wynikami

In [ ]:
df_results = pd.DataFrame(results_ex1)

# Uporządkowanie kolejności kolumn dla lepszej czytelności
cols = ['Detektor', 'Parametry', 'Prawdziwe twarze w GT', 'Wykryte obiekty',
        'Trafienia (TP)', 'Fałszywe (FP)', 'Przeoczone (FN)', 'Precyzja (%)', 'Czułość/Recall (%)']
df_results = df_results.reindex(columns=cols)

# Wyświetlenie wyrenderowanej tabelki HTML w notatniku
display(df_results)

Po analizie wyników detekcji na obrazie i danych z powyższej tabelki można wysnuć parę obserwacji:
- Kaskada Haara jako jedyna miała tendencję do zwracania wielu FP - jej wskaźnik precyzji jest niski w porównaniu z innymi detektorami. Jednakże warto zauważyć, że zwracając wiele wyników osiągnęła lepszą czułość niż HOG czy MMOD. W natłoku detekcji oprócz masy FP było też dużo Trafień.
- Detektory z biblioteki Dlib (HOG i MMOD) charakteryzują się bardzo niskim wskaźnikiem czułości, lecz nadrabiają stuprocentową precyzją detekcji. Można stwierdzić, że są dość ostrożne w działaniu.
- InsightFace przewyższa w działaniu sieć MMOD - ma bardzo dobrą czułość przy równoczesnej stuprocentowej precyzji.

Z tych wniosków można stwierdzić, że najlepszym detektorem jest InsightFace. Jednakże warto pamiętać, że nie był w stanie wykryć wszystkich twarzy na obrazie. Sam obraz jest 'trudny' - twarze znajdują się w głębi obrazu i są większości przykryte lub lekko rozmazane. Dlatego wynik 15 trafień na 19 wszystkich twarzy jest bardzo dobry. 

## Zadanie 2 - więcej obrazów testowych

In [ ]:
results_ex2_img1 = []
results_ex2_img2 = []
results_ex2_img3 = []

image1_ex2 = "0--Parade/0_Parade_Parade_0_873.jpg"
image2_ex2 = "23--Shoppers/23_Shoppers_Shoppers_23_561.jpg"
image3_ex2 = "49--Greeting/49_Greeting_peoplegreeting_49_218.jpg"

gt_ex2_img1 = gt_all_images.get(image1_ex2, [])
gt_ex2_img2 = gt_all_images.get(image2_ex2, [])
gt_ex2_img3 = gt_all_images.get(image3_ex2, [])

### Obrazy testowe

In [ ]:
img_1 = cv2.imread("0_Parade_Parade_0_873.jpg")
# przygotowanie obrazów: monochromatycznego i RGB (cv2.imread() zwraca obraz w formacie BGR - inna kolejność składowych)
img_gray_1 = cv2.cvtColor(img_1, cv2.COLOR_BGR2GRAY)
img_rgb_1 = cv2.cvtColor(img_1, cv2.COLOR_BGR2RGB)

display(Image.fromarray(img_rgb_1))  # display() wymaga obrazu RGB (w przeciwieństwie do cv2.imshow(), która wymaga BGR)

In [ ]:
img_2 = cv2.imread("23_Shoppers_Shoppers_23_561.jpg")
# przygotowanie obrazów: monochromatycznego i RGB (cv2.imread() zwraca obraz w formacie BGR - inna kolejność składowych)
img_gray_2 = cv2.cvtColor(img_2, cv2.COLOR_BGR2GRAY)
img_rgb_2 = cv2.cvtColor(img_2, cv2.COLOR_BGR2RGB)

display(Image.fromarray(img_rgb_2))  # display() wymaga obrazu RGB (w przeciwieństwie do cv2.imshow(), która wymaga BGR)

In [ ]:
img_3 = cv2.imread("49_Greeting_peoplegreeting_49_218.jpg")
# przygotowanie obrazów: monochromatycznego i RGB (cv2.imread() zwraca obraz w formacie BGR - inna kolejność składowych)
img_gray_3 = cv2.cvtColor(img_3, cv2.COLOR_BGR2GRAY)
img_rgb_3 = cv2.cvtColor(img_3, cv2.COLOR_BGR2RGB)

display(Image.fromarray(img_rgb_3))  # display() wymaga obrazu RGB (w przeciwieństwie do cv2.imshow(), która wymaga BGR)

### Kaskada Haara

In [ ]:
haar_detector = cv2.CascadeClassifier("haarcascade_frontalface_default.xml")
haar_results_1 = haar_detector.detectMultiScale(img_gray_1, scaleFactor=1.05, minNeighbors=5, minSize=(30, 30), flags=cv2.CASCADE_SCALE_IMAGE)
res_haar_1 = evaluate_with_gt(haar_results_1, gt_ex2_img1, detector_type=DetectorTypes.HAAR)
res_haar_1["Detektor"] = "OpenCV (Haar)"
res_haar_1["Parametry"] = "scaleFactor=1.05, minNeighbors=5, minSize=(30, 30)"
results_ex2_img1.append(res_haar_1)
results_all.append(res_haar_1)

haar_results_2 = haar_detector.detectMultiScale(img_gray_2, scaleFactor=1.05, minNeighbors=5, minSize=(30, 30), flags=cv2.CASCADE_SCALE_IMAGE)
res_haar_2 = evaluate_with_gt(haar_results_2, gt_ex2_img2, detector_type=DetectorTypes.HAAR)
res_haar_2["Detektor"] = "OpenCV (Haar)"
res_haar_2["Parametry"] = "scaleFactor=1.05, minNeighbors=5, minSize=(30, 30)"
results_ex2_img2.append(res_haar_2)
results_all.append(res_haar_2)

haar_results_3 = haar_detector.detectMultiScale(img_gray_3, scaleFactor=1.05, minNeighbors=5, minSize=(30, 30), flags=cv2.CASCADE_SCALE_IMAGE)
res_haar_3 = evaluate_with_gt(haar_results_3, gt_ex2_img3, detector_type=DetectorTypes.HAAR)
res_haar_3["Detektor"] = "OpenCV (Haar)"
res_haar_3["Parametry"] = "scaleFactor=1.05, minNeighbors=5, minSize=(30, 30)"
results_ex2_img3.append(res_haar_3)
results_all.append(res_haar_3)

In [ ]:
img_haar = img_rgb_1.copy()
for (x, y, w, h) in haar_results_1:
    cv2.rectangle(img_haar, (x, y), (x + w, y + h), (0, 255, 0), 2)
display(Image.fromarray(img_haar))

In [ ]:
img_haar = img_rgb_2.copy()
for (x, y, w, h) in haar_results_2:
    cv2.rectangle(img_haar, (x, y), (x + w, y + h), (0, 255, 0), 2)
display(Image.fromarray(img_haar))

In [ ]:
img_haar = img_rgb_3.copy()
for (x, y, w, h) in haar_results_3:
    cv2.rectangle(img_haar, (x, y), (x + w, y + h), (0, 255, 0), 2)
display(Image.fromarray(img_haar))

### Histogram zorientowanych gradientów (HOG) z maszyną wektorów nośnych (SVM)

In [ ]:
hog_svm_detector = dlib.get_frontal_face_detector()

hog_svm_results_1 = hog_svm_detector(img_rgb_1, 1)
res_dlib_hog_1 = evaluate_with_gt(hog_svm_results_1, gt_ex2_img1, detector_type=DetectorTypes.HOG_SVM)
res_dlib_hog_1["Detektor"] = "Dlib (HOG)"
res_dlib_hog_1["Parametry"] = "upsample=1, scale=1"
results_ex2_img1.append(res_dlib_hog_1)
results_all.append(res_dlib_hog_1)

hog_svm_results_2 = hog_svm_detector(img_rgb_2, 1)
res_dlib_hog_2 = evaluate_with_gt(hog_svm_results_2, gt_ex2_img2, detector_type=DetectorTypes.HOG_SVM)
res_dlib_hog_2["Detektor"] = "Dlib (HOG)"
res_dlib_hog_2["Parametry"] = "upsample=1, scale=1"
results_ex2_img2.append(res_dlib_hog_2)
results_all.append(res_dlib_hog_2)

hog_svm_results_3 = hog_svm_detector(img_rgb_3, 1)
res_dlib_hog_3 = evaluate_with_gt(hog_svm_results_3, gt_ex2_img3, detector_type=DetectorTypes.HOG_SVM)
res_dlib_hog_3["Detektor"] = "Dlib (HOG)"
res_dlib_hog_3["Parametry"] = "upsample=1, scale=1"
results_ex2_img3.append(res_dlib_hog_3)
results_all.append(res_dlib_hog_3)

In [ ]:
img_hog_svm = img_rgb_1.copy()
for rect in hog_svm_results_1:
    cv2.rectangle(img_hog_svm, (rect.left(), rect.top()), (rect.right(), rect.bottom()), (0, 255, 0), 2)
display(Image.fromarray(img_hog_svm))

In [ ]:
img_hog_svm = img_rgb_2.copy()
for rect in hog_svm_results_2:
    cv2.rectangle(img_hog_svm, (rect.left(), rect.top()), (rect.right(), rect.bottom()), (0, 255, 0), 2)
display(Image.fromarray(img_hog_svm))

In [ ]:
img_hog_svm = img_rgb_3.copy()
for rect in hog_svm_results_3:
    cv2.rectangle(img_hog_svm, (rect.left(), rect.top()), (rect.right(), rect.bottom()), (0, 255, 0), 2)
display(Image.fromarray(img_hog_svm))

### Splotowa sieć neuronowa (CNN) z biblioteki dlib

In [ ]:
cnn1_detector = dlib.cnn_face_detection_model_v1('mmod_human_face_detector.dat')

cnn1_results_1 = cnn1_detector(img_rgb_1, 1)
res_dlib_mmod_1 = evaluate_with_gt(cnn1_results_1, gt_ex2_img1, detector_type=DetectorTypes.MMOD)
res_dlib_mmod_1["Detektor"] = "Dlib (MMOD)"
res_dlib_mmod_1["Parametry"] = "upsample=1, scale=1"
results_ex2_img1.append(res_dlib_mmod_1)
results_all.append(res_dlib_mmod_1)

cnn1_results_2 = cnn1_detector(img_rgb_2, 1)
res_dlib_mmod_2 = evaluate_with_gt(cnn1_results_2, gt_ex2_img2, detector_type=DetectorTypes.MMOD)
res_dlib_mmod_2["Detektor"] = "Dlib (MMOD)"
res_dlib_mmod_2["Parametry"] = "upsample=1, scale=1"
results_ex2_img2.append(res_dlib_mmod_2)
results_all.append(res_dlib_mmod_2)

cnn1_results_3 = cnn1_detector(img_rgb_3, 1)
res_dlib_mmod_3 = evaluate_with_gt(cnn1_results_3, gt_ex2_img3, detector_type=DetectorTypes.MMOD)
res_dlib_mmod_3["Detektor"] = "Dlib (MMOD)"
res_dlib_mmod_3["Parametry"] = "upsample=1, scale=1"
results_ex2_img3.append(res_dlib_mmod_3)
results_all.append(res_dlib_mmod_3)

In [ ]:
img_cnn1 = img_rgb_1.copy()
for res in cnn1_results_1:
    rect = res.rect
    cv2.rectangle(img_cnn1, (rect.left(), rect.top()), (rect.right(), rect.bottom()), (0, 255, 0), 2)
display(Image.fromarray(img_cnn1))

In [ ]:
img_cnn1 = img_rgb_2.copy()
for res in cnn1_results_2:
    rect = res.rect
    cv2.rectangle(img_cnn1, (rect.left(), rect.top()), (rect.right(), rect.bottom()), (0, 255, 0), 2)
display(Image.fromarray(img_cnn1))

In [ ]:
img_cnn1 = img_rgb_3.copy()
for res in cnn1_results_3:
    rect = res.rect
    cv2.rectangle(img_cnn1, (rect.left(), rect.top()), (rect.right(), rect.bottom()), (0, 255, 0), 2)
display(Image.fromarray(img_cnn1))

### InsightFace - inna sieć neuronowa

In [ ]:
model_name = 'buffalo_s'  # model: small (_s), medium (_m) lub large (_l)
insf_detector = insightface.app.FaceAnalysis(name=model_name, root='insightface',
                                             allowed_modules=['detection'], providers=['CPUExecutionProvider'])
insf_detector.prepare(ctx_id=0, det_size=(1024, 1024))

In [ ]:
insf_results_1 = insf_detector.get(img_1)
res_insf_1 = evaluate_with_gt(insf_results_1, gt_ex2_img1, detector_type=DetectorTypes.INSIGHTFACE)
res_insf_1["Detektor"] = "InsightFace"
res_insf_1["Parametry"] = "det_size=(1024, 1024)"
results_ex2_img1.append(res_insf_1)
results_all.append(res_insf_1)

insf_results_2 = insf_detector.get(img_2)
res_insf_2 = evaluate_with_gt(insf_results_2, gt_ex2_img2, detector_type=DetectorTypes.INSIGHTFACE)
res_insf_2["Detektor"] = "InsightFace"
res_insf_2["Parametry"] = "det_size=(1024, 1024)"
results_ex2_img2.append(res_insf_2)
results_all.append(res_insf_2)

insf_results_3 = insf_detector.get(img_3)
res_insf_3 = evaluate_with_gt(insf_results_3, gt_ex2_img3, detector_type=DetectorTypes.INSIGHTFACE)
res_insf_3["Detektor"] = "InsightFace"
res_insf_3["Parametry"] = "det_size=(1024, 1024)"
results_ex2_img3.append(res_insf_3)
results_all.append(res_insf_3)

In [ ]:
img_insf = img_rgb_1.copy()
for res in insf_results_1:
    rect = res.bbox.round().astype(int)
    cv2.rectangle(img_insf, (rect[0], rect[1]), (rect[2], rect[3]), (0, 255, 0), 2)
display(Image.fromarray(img_insf))

In [ ]:
img_insf = img_rgb_2.copy()
for res in insf_results_2:
    rect = res.bbox.round().astype(int)
    cv2.rectangle(img_insf, (rect[0], rect[1]), (rect[2], rect[3]), (0, 255, 0), 2)
display(Image.fromarray(img_insf))

In [ ]:
img_insf = img_rgb_3.copy()
for res in insf_results_3:
    rect = res.bbox.round().astype(int)
    cv2.rectangle(img_insf, (rect[0], rect[1]), (rect[2], rect[3]), (0, 255, 0), 2)
display(Image.fromarray(img_insf))

### Wyniki

In [ ]:
df_results_1 = pd.DataFrame(results_ex2_img1)
df_results_2 = pd.DataFrame(results_ex2_img2)
df_results_3 = pd.DataFrame(results_ex2_img3)

# Uporządkowanie kolejności kolumn dla lepszej czytelności
cols = ['Detektor', 'Parametry', 'Prawdziwe twarze w GT', 'Wykryte obiekty',
        'Trafienia (TP)', 'Fałszywe (FP)', 'Przeoczone (FN)', 'Precyzja (%)', 'Czułość/Recall (%)']
df_results_1 = df_results_1.reindex(columns=cols)
df_results_2 = df_results_2.reindex(columns=cols)
df_results_3 = df_results_3.reindex(columns=cols)

# Wyświetlenie wyrenderowanej tabelki HTML w notatniku
print("Pierwszy obraz")
display(df_results_1)
print("Drugi obraz")
display(df_results_2)
print("Trzeci obraz")
display(df_results_3)

#### Pierwszy obraz - tłum ludzi:
Ewidentnie można stwierdzić, że znów InsightFace wygrywa w tej kategorii 'trudnych' obrazów. HOG poradziło sobie najgorzej, mając okropny wskaźnik czułości. Haar też wypadł słabo. MMOD też nie jest najlepszy, lecz jest o wiele lepszy od Haara i HOG-a.

#### Drugi obraz - opakowania produktów:
Tutaj każdy z detektorów miał podobne wyniki. Haar lekko przestrzelił z dwoma detekcjami. HOG i MMOD mają identyczne wyniki, a InsightFace przebija resztę o dwa dobre Trafienia.

#### Trzeci obraz - znaki wodne, parada i dużo bluru
Najcięższy obraz do detekcji - dużo elementów przykrywających twarze oraz rozmazujące obraz. Haar w tych warunkach poradził sobie najgorzej - pomylił znak wodny z twarzą i znalazł poprawnie jedną twarz.
HOG i MMOD mają identyczne wyniki - wykyły dość oczywiste twarze. InsightFace poradził sobie najlepiej, ale nadal nie idealnie - nie dał rady znaleźć połowy twarzy na obrazie.

### Wnioski
Z analizy tych trzech obrazów można umocnić się w poprzednich stwierdzeniacj i obserwacjach. InsightFace jest najlepszy - znajduje najwięcej twarzy, często mocno przykryte lub rozmazane. Radzi sobie bardzo dobrze z 'trudnymi' zdjęciami. Jego wadą jest to, że detekcje zajmuje trochę czasu. HOG i MMOD mają porównywalne wyniki dla większoći obrazów, lecz MMOD lepiej się sprawdza w obrazach z tłumami. Natomiast Haar jest tylko efektywny dla łatwych obrazów oraz ma tendencję do naddetekcji - jako jednyny w większości nie osiąga 100 % precyzji. Jednakże, czasami jest lepszy niż HOG i jest bardzo szybki.

## Zadanie 3 - podsumowanie dotychczasowych testów

In [319]:
df_results_all = pd.DataFrame(results_all)

# Uporządkowanie kolejności kolumn dla lepszej czytelności
columns_to_sum = ['Wykryte obiekty', 'Prawdziwe twarze w GT', 'Trafienia (TP)', 'Fałszywe (FP)', 'Przeoczone (FN)']

df_global = df_results_all.groupby('Detektor')[columns_to_sum].sum().reset_index()

df_global['Precyzja (%)'] = df_global.apply(
    lambda row: round((row['Trafienia (TP)'] / row['Wykryte obiekty']) * 100, 2) if row['Wykryte obiekty'] > 0 else 0,
    axis=1
)

df_global['Czułość/Recall (%)'] = df_global.apply(
    lambda row: round((row['Trafienia (TP)'] / row['Prawdziwe twarze w GT']) * 100, 2) if row['Prawdziwe twarze w GT'] > 0 else 0,
    axis=1
)

# 3. Wyświetlenie globalnej tabeli wymaganej w pkt 3
print("Zestawienie globalne (łącznie dla wszystkich badanych obrazów z pkt 1 i 2):")
display(df_global)

Zestawienie globalne (łącznie dla wszystkich badanych obrazów z pkt 1 i 2):


,Detektor,Wykryte obiekty,Prawdziwe twarze w GT,Trafienia (TP),Fałszywe (FP),Przeoczone (FN),Precyzja (%),Czułość/Recall (%)
0,Dlib (HOG),16,181,16,0,165,100.00,8.84
1,Dlib (MMOD),33,181,33,0,148,100.00,18.23
2,InsightFace,106,181,104,2,77,98.11,57.46
3,OpenCV (Haar),40,181,22,18,159,55.00,12.15


Tak jak w poprzednich punktach wnioski są podobne - te dane jeszcze bardziej je utwierdzają. InsightFace najlepiej się sprawdza, chodź nie jest idealny. MMOD jest o wiele gorszy niż InsigtFace, lecz jest lepszy od HOG. Na papierze Haar jest lepszy niż HOG, ale większość detekcji jest fałszywa.

## Zadanie 4 - skalowanie obrazów do detekcji

Do zbadania wpływu skalowania rozmiaru obrazu na właściwości detekcji użyje 'trudnego' obrazu z dużą liczbą twarzy. Został on wykorzystany w zadaniu 2 jako pierwszy obraz.

In [ ]:
results_ex4 = []

### Histogram zorientowanych gradientów (HOG) z maszyną wektorów nośnych (SVM)

In [ ]:
hog_svm_detector = dlib.get_frontal_face_detector()

In [ ]:
# Dla porównania później w tabeli
results_ex4.append(res_dlib_hog_1)

hog_svm_scale_0 = hog_svm_detector(img_rgb_1, 0)
res_dlib_hog_scale_0 = evaluate_with_gt(hog_svm_scale_0, gt_ex2_img1, detector_type=DetectorTypes.HOG_SVM)
res_dlib_hog_scale_0["Detektor"] = "Dlib (HOG)"
res_dlib_hog_scale_0["Parametry"] = "upsample=1, scale=0"
results_ex4.append(res_dlib_hog_scale_0)

hog_svm_scale_2 = hog_svm_detector(img_rgb_1, 2)
res_dlib_hog_scale_2 = evaluate_with_gt(hog_svm_scale_2, gt_ex2_img1, detector_type=DetectorTypes.HOG_SVM)
res_dlib_hog_scale_2["Detektor"] = "Dlib (HOG)"
res_dlib_hog_scale_2["Parametry"] = "upsample=1, scale=2"
results_ex4.append(res_dlib_hog_scale_2)

In [ ]:
img_hog_svm = img_rgb_1.copy()
for rect in hog_svm_scale_0:
    cv2.rectangle(img_hog_svm, (rect.left(), rect.top()), (rect.right(), rect.bottom()), (0, 255, 0), 2)
display(Image.fromarray(img_hog_svm))

In [ ]:
img_hog_svm = img_rgb_1.copy()
for rect in hog_svm_scale_2:
    cv2.rectangle(img_hog_svm, (rect.left(), rect.top()), (rect.right(), rect.bottom()), (0, 255, 0), 2)
display(Image.fromarray(img_hog_svm))

### Splotowa sieć neuronowa (CNN) z biblioteki dlib

In [ ]:
cnn1_detector = dlib.cnn_face_detection_model_v1('mmod_human_face_detector.dat')

In [ ]:
# Dla porównania później w tabeli
results_ex4.append(res_dlib_mmod_1)

cnn1_results_scale_0 = cnn1_detector(img_rgb_1, 0)
res_dlib_mmod_scale_0 = evaluate_with_gt(cnn1_results_scale_0, gt_ex2_img1, detector_type=DetectorTypes.MMOD)
res_dlib_mmod_scale_0["Detektor"] = "Dlib (MMOD)"
res_dlib_mmod_scale_0["Parametry"] = "upsample=1, scale=0"
results_ex4.append(res_dlib_mmod_scale_0)

cnn1_results_scale_2 = cnn1_detector(img_rgb_1, 2)
res_dlib_mmod_scale_2 = evaluate_with_gt(cnn1_results_scale_2, gt_ex2_img1, detector_type=DetectorTypes.MMOD)
res_dlib_mmod_scale_2["Detektor"] = "Dlib (MMOD)"
res_dlib_mmod_scale_2["Parametry"] = "upsample=1, scale=2"
results_ex4.append(res_dlib_mmod_scale_2)

In [ ]:
img_cnn1 = img_rgb_1.copy()
for res in cnn1_results_scale_0:
    rect = res.rect
    cv2.rectangle(img_cnn1, (rect.left(), rect.top()), (rect.right(), rect.bottom()), (0, 255, 0), 2)
display(Image.fromarray(img_cnn1))

In [ ]:
img_cnn1 = img_rgb_1.copy()
for res in cnn1_results_scale_2:
    rect = res.rect
    cv2.rectangle(img_cnn1, (rect.left(), rect.top()), (rect.right(), rect.bottom()), (0, 255, 0), 2)
display(Image.fromarray(img_cnn1))

### InsightFace - inna sieć neuronowa

In [ ]:
model_name = 'buffalo_s'  # model: small (_s), medium (_m) lub large (_l)
insf_detector = insightface.app.FaceAnalysis(name=model_name, root='insightface',
                                             allowed_modules=['detection'], providers=['CPUExecutionProvider'])
insf_detector.prepare(ctx_id=0, det_size=(512, 512))

In [ ]:
# Dla porównania później w tabeli
results_ex4.append(res_insf_1)

insf_results_size_512 = insf_detector.get(img_1)
res_insf_size_512 = evaluate_with_gt(insf_results_size_512, gt_ex2_img1, detector_type=DetectorTypes.INSIGHTFACE)
res_insf_size_512["Detektor"] = "InsightFace"
res_insf_size_512["Parametry"] = "det_size=(512, 512)"
results_ex4.append(res_insf_size_512)

In [ ]:
img_insf = img_rgb_1.copy()
for res in insf_results_size_512:
    rect = res.bbox.round().astype(int)
    cv2.rectangle(img_insf, (rect[0], rect[1]), (rect[2], rect[3]), (0, 255, 0), 2)
display(Image.fromarray(img_insf))

In [ ]:
model_name = 'buffalo_s'  # model: small (_s), medium (_m) lub large (_l)
insf_detector = insightface.app.FaceAnalysis(name=model_name, root='insightface',
                                             allowed_modules=['detection'], providers=['CPUExecutionProvider'])
insf_detector.prepare(ctx_id=0, det_size=(2048, 2048))

In [ ]:
insf_results_size_2048 = insf_detector.get(img_1)
res_insf_size_2048 = evaluate_with_gt(insf_results_size_2048, gt_ex2_img1, detector_type=DetectorTypes.INSIGHTFACE)
res_insf_size_2048["Detektor"] = "InsightFace"
res_insf_size_2048["Parametry"] = "det_size=(2048, 2048)"
results_ex4.append(res_insf_size_2048)

In [ ]:
img_insf = img_rgb_1.copy()
for res in insf_results_size_2048:
    rect = res.bbox.round().astype(int)
    cv2.rectangle(img_insf, (rect[0], rect[1]), (rect[2], rect[3]), (0, 255, 0), 2)
display(Image.fromarray(img_insf))

### Wyniki

In [320]:
df_results_ex4 = pd.DataFrame(results_ex4)

cols = ['Detektor', 'Parametry', 'Prawdziwe twarze w GT', 'Wykryte obiekty',
        'Trafienia (TP)', 'Fałszywe (FP)', 'Przeoczone (FN)', 'Precyzja (%)', 'Czułość/Recall (%)']
df_results_ex4 = df_results_ex4.reindex(columns=cols)

display(df_results_ex4)

,Detektor,Parametry,Prawdziwe twarze w GT,Wykryte obiekty,Trafienia (TP),Fałszywe (FP),Przeoczone (FN),Precyzja (%),Czułość/Recall (%)
0,Dlib (HOG),"upsample=1, scale=1",142,4,4,0,138,100.00,2.82
1,Dlib (HOG),"upsample=1, scale=0",142,2,2,0,140,100.00,1.41
2,Dlib (HOG),"upsample=1, scale=2",142,15,14,1,128,93.33,9.86
3,Dlib (MMOD),"upsample=1, scale=1",142,19,19,0,123,100.00,13.38
4,Dlib (MMOD),"upsample=1, scale=0",142,4,4,0,138,100.00,2.82
5,Dlib (MMOD),"upsample=1, scale=2",142,43,42,1,100,97.67,29.58
6,InsightFace,"det_size=(1024, 1024)",142,80,78,2,64,97.50,54.93
7,InsightFace,"det_size=(512, 512)",142,32,29,3,113,90.62,20.42
8,InsightFace,"det_size=(2048, 2048)",142,96,95,1,47,98.96,66.90


Zmiana parametrów skalowania obrazu dla detektorów w Dlib i zmiana rozmiaru obrazu w detektorze InsightFace prowadzie do ciekawych rezultatów. Zmniejsze skalowania do 0 i rozmiaru obrazu do 512x512 spowodowało ewidentny spadek jakości detekcji dla wyszystkich detektorów w porównaniu do ich bazowych parametrów. W szczególności dla detektorów z Dlib skala 0 prawie całkowicie nulifikuje jakiekolwiek zdoloności detekcji twarzy. InsightFace jeszce sobie radzi, ale nienajlepiej porównując do bazy.

Natomiast sytuacja dla skalowania 2 i rozmiaru 2048x2048 diametralnie się zmienia. Wszystkie detekoty osiągają lepsze wyniki niż dla bazowych parametrów. Szczególnie to widać dla MMOD, gdzie zmiana na dwukrotne skalowanie dała około dwukrotny wzrost Trafień. HOG też się poprawił o około 7 pkt. procentowych w kwestii czułości. InsightFace osiągnęło jeszcze lepsze wyniki, dochodząc do prawie 67 % czułości.

Jasno wynika, że zmiana skalowania tudzież rozmiaru obrazu diametralnie wpyływa na działanie detektorów. Im większy obraz, tym lepsza detekcja dla wszystkich detektorów. Najlepiej na tym wychodzi MMOD. Jednakże mniejszy obraz powoduje całkowity spadek w detekcji dla HOG i MMOD - tylko InsightFace zachowuje jako takie możliwości detekcji.